In [9]:
from colorama import Fore, Back, Style

def PP_error(msg: str):
    print(Fore.RED + Style.BRIGHT + msg + Fore.RESET + Style.RESET_ALL)


def PP_warning(msg: str):
    print(Fore.YELLOW + msg + Fore.RESET)


def PP_good(msg: str):
    print(Fore.GREEN + Style.BRIGHT + msg + Fore.RESET + Style.RESET_ALL)

def PP_warning_with_error(msg: str):
    final_string = ''
    current_string = msg
    s = len(msg.split('e<'))
    while s > 1:
        split = current_string.split('e<')
        s = len(split)
        current_string = current_string.removeprefix(split[0] + 'e<')
        final_string += Fore.RESET + Style.RESET_ALL + Fore.YELLOW + split[0]
        if s == 1:
            break

        split = current_string.split('>')
        s = len(split)
        current_string = current_string.removeprefix(split[0] + '>')
        final_string += Fore.RESET + Style.BRIGHT + Fore.RED + split[0]

    final_string += Fore.RESET + Style.RESET_ALL
    print(final_string)


In [10]:
import os
import re
import time

from skimage import io
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.transforms import functional
import torchvision.utils
from torchvision.utils import make_grid


class PGMDataset(Dataset):
    """ PGM Dataset """

    def __init__(self, csv_file='dataset.csv', pgm_dir='Data/', check_dataset_integrity=True, transform=None):
        """
        Arguments:
            :param csv_file (string): CSV with [file_name, labels]
            :param pgm_dir (string): Directory with PGMs
            :param transform (callable, optional): Transform to be applied

            Note: csv_file example is:
                snow_1,0,0,0,1,0

                sun_1,1,0,0,0,0
        """
        self.possible_labels = ['sun', 'rain', 'fog', 'snow', 'synth']
        self.pgm_dir = pgm_dir
        self.csv_file = csv_file
        self.transform = transform

        self.dataset = pd.read_csv(os.path.join('Dataset',self.csv_file))
        self.len = len(self.dataset)

        self.weather_inds = []
        self.weather_len = 0
        self.synth_inds = []
        self.synth_len = 0

        self.dataset_balanced = True

        if not check_dataset_integrity:
            return

        PP_warning("Checking dataset...")
        incorrect_files_number = 0
        # Remove from dataset all non-existing PGM written in the csv
        for i in range(0, self.len):
            if i == self.len:
                break
            if not os.path.exists(os.path.join(self.pgm_dir, self.dataset.iloc[i, 0])):
                incorrect_files_number += 1
                self.dataset = self.dataset.drop([i])
                self.len -= 1

        if incorrect_files_number > 0:
            PP_error("{} missing files in PGMs folder!".format(incorrect_files_number))
            PP_error("Those files will not be considered!")
        else:
            PP_good("CSV and PGM are correct.")

        # Keep track of the indexes of synth and weather pgm
        for i in range(0, self.len):
            if self.dataset.iloc[i, -1] == 1:
                self.synth_inds.append(i)
            else:
                self.weather_inds.append(i)

        # It's better to have an balanced dataset, notify if it's not
        self.weather_len = len(self.weather_inds)
        self.synth_len = len(self.synth_inds)
        if self.weather_len != self.synth_len:
            self.dataset_balanced = False
            PP_error("The dataset is unbalanced!\nFor better training, balance the dataset.")
            PP_error("Found {} real weather and {} synth pgm!".format(len(self.weather_inds), len(self.synth_inds)))
        else:
            PP_good("Dataset is balanced!")

    def is_filename_correct(self, filename) -> bool:
        matches = re.match(r'(.+)_(\d+)\.png', filename)

        if matches is None:
            return False

        weather = matches.group(1)
        number = int(matches.group(2))

        if weather not in self.possible_labels:
            return False
        return True

    def __len__(self):
        # If is balanced, return len / 2, as a sample is [Weather_PGM, Synth_PGM, ...]
        if self.dataset_balanced:
            return int(self.len / 2)
        # If not balanced, return the max len of PGM we have. __getitem__() will fill the gap reusing past PGMs
        return int(max(self.weather_len, self.synth_len))

    def __getitem__(self, ind):
        """
        :param ind: index
        :return: ([weather_pgm], [synth_pgm],
        """
        if torch.is_tensor(ind):
            ind = ind.tolist()

        weather_pgm_index = self.weather_inds[ind % self.weather_len]
        synth_pgm_index = self.synth_inds[ind % self.synth_len]

        #if self.weather_len < self.synth_len:
        #    weather_pgm_index = ind % self.weather_len
        #elif self.synth_len < self.weather_len:
        #    synth_pgm_index = ind % self.synth_len

        weather_pgm_name = os.path.join(self.pgm_dir, self.dataset.iloc[weather_pgm_index, 0])
        weather_pgm = io.imread(weather_pgm_name)
        weather_labels = self.dataset.iloc[weather_pgm_index, 1:]
        weather_labels = np.asarray(weather_labels, dtype=float)

        synth_pgm_name = os.path.join(self.pgm_dir, self.dataset.iloc[synth_pgm_index, 0])
        synth_pgm = io.imread(synth_pgm_name)
        synth_label = [0.0, 0.0, 0.0, 0.0, 1.0]
        synth_label = np.asarray(synth_label, dtype=float)

        sample = {'weather_pgm': weather_pgm, 'synth_pgm': synth_pgm,
                  'weather_label': weather_labels, 'synth_label': synth_label}

        if self.transform:
            sample['weather_pgm'] = self.transform(sample['weather_pgm'])
            sample['synth_pgm'] = self.transform(sample['synth_pgm'])
        return sample


def get_dataloader(config, mode='train', num_workers=1):
    transform = transforms.Compose([transforms.ToTensor(),
                                    #transforms.CenterCrop((min(config.image_size[0], 32),
                                    #                       min(config.image_size[1], 1024))),
                                    transforms.Resize(config.image_size,
                                                      interpolation=functional.InterpolationMode.NEAREST,
                                                      antialias=False)
                                  ])
    dataset = PGMDataset(config.csv_file, config.pgm_dir, True, transform)
    return DataLoader(dataset=dataset,
                      batch_size=config.batch_size,
                      shuffle=(mode == 'train'),
                      num_workers=num_workers,
                      drop_last=True)


def show_sample(sample):
    w_img = sample['weather_pgm']
    s_img = sample['synth_pgm']

    if len(w_img.shape) == 4:
        w_img = w_img[0]
    if len(s_img.shape) == 4:
        s_img = s_img[0]

    T = transforms.ToPILImage()
    if torch.is_tensor(w_img):
        w_img = T(w_img)
    if torch.is_tensor(s_img):
        s_img = T(s_img)

    label = sample['weather_label']
    if len(label.shape) == 2:
        label = label[0]

    title = ''

    if label[0].item() == 1.0:
        title += 'SUN '
    if label[1].item() == 1.0:
        title += 'RAIN '
    if label[2].item() == 1.0:
        title += 'FOG '
    if label[3].item() == 1.0:
        title += 'SNOW'

    fig, axis = plt.subplots(nrows=2, ncols=1, figsize=(12, 2), tight_layout=True)

    axis[0].axis('off')
    axis[1].axis('off')
    axis[0].set_title(title)
    axis[1].set_title('SYNTH')
    axis[0].imshow(w_img, cmap='afmhot', interpolation='none')
    axis[1].imshow(s_img, cmap='afmhot', interpolation='none')
    plt.show()


def show_pgm(pgm):
    if len(pgm.shape) == 4:
        pgm = pgm[0]
    T = transforms.ToPILImage()
    if torch.is_tensor(pgm):
        pgm = T(pgm)
    plt.axis('off')
    plt.imshow(pgm, cmap='afmhot', interpolation='none')
    plt.show()


def show_batch(batch):
    fig = plt.figure(figsize=(2, 8))
    to_pil = transforms.ToPILImage()
    for i in range(1, 5):
        img = to_pil(batch['weather_pgm'][i])
        fig.add_subplot(8, 1, i)
        label = batch['weather_label'][i]

        title = ''
        if label[0] == 1.0:
            title += 'SUN '
        if label[1] == 1.0:
            title += 'RAIN '
        if label[2] == 1.0:
            title += 'FOG '
        if label[3] == 1.0:
            title += 'SNOW '
        if len(label[i]) == 5 and label[4] == 1.0:
            title += 'SYNTH '
        plt.title(title)
        plt.imshow(img, cmap='gray', interpolation='none')
    for i in range(1, 5):
        img = to_pil(batch['synth_pgm'][i])
        fig.add_subplot(8, 1, i+4)
        label = batch['synth_label'][i]

        title = 'SYNTH '
        plt.title(title)
        plt.imshow(img, cmap='gray', interpolation='none')
    plt.show()

def save_synth_and_generated(synth_pgm, generated_pgm, target_labels, masks, name, nrows):

    to_pil = transforms.ToPILImage()
    fig = plt.figure(figsize=(15, 25))
    fig_number = 1
    for i in range(1, 8, 2):
        fig.add_subplot(12, 1, fig_number)
        img = to_pil(synth_pgm[i])
        title = ''
        if i > 3:
            title = 'REAL '
        if target_labels[i][0] == 1.0:
            title += 'SUN '
        if target_labels[i][1] == 1.0:
            title += 'RAIN '
        if target_labels[i][2] == 1.0:
            title += 'FOG '
        if target_labels[i][3] == 1.0:
            title += 'SNOW '
        if len(target_labels[i]) == 5 and target_labels[i][4] == 1.0:
            title += 'SYNTH '
        plt.axis('off')
        plt.title(title, fontdict={'color':'red'})
        plt.subplots_adjust(hspace=0)
        plt.imshow(img, cmap='gray', interpolation='none')
        
        fig.add_subplot(12, 1, fig_number + 1)
        img = to_pil(masks[i])
        plt.axis('off')
        plt.imshow(img, cmap='gray', interpolation='none')        
        
        fig.add_subplot(12, 1, fig_number + 2)
        img = to_pil(generated_pgm[i])
        plt.axis('off')
        plt.imshow(img, cmap='gray', interpolation='none')
        fig_number += 3
        

    # plt.show()
    plt.savefig(name + '.png')
    # batch = torch.zeros_like(synth_pgm)
    # for i in range(0, synth_pgm.shape[0], 2):
    #     batch[i] = synth_pgm[i]
    #     batch[i + 1] = generated_pgm[i]
    # grid = make_grid(batch.detach().cpu(), nrow=nrows)
    # torchvision.utils.save_image(grid, name + '.png')


In [11]:
import torch
import torch.nn as nn
import math
from torch.nn.utils.parametrizations import spectral_norm
import torch.nn.functional as F
import numpy as np

class GaussianNoise(nn.Module):
    """Gaussian noise regularizer.
    """

    def __init__(self, sigma=0.1):
        super().__init__()
        self.sigma = sigma
        self.noise = torch.tensor(0).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    
    def forward(self,x):
        out = torch.randn_like(x) * 0.05
        out = x + out
        out = F.relu(out)
        out /= (out.sum(dim=1, keepdim=True) + 1e-8)
        return out

    #def forward(self, x):
    #    sampled_noise = self.noise.repeat(*x.size()).float().normal_() * self.sigma
    #    x = x + sampled_noise
    #    return x 

class RCL(nn.Module):
    def __init__(self, inplanes, steps=4):
        super(RCL, self).__init__()
        self.conv = nn.Conv2d(inplanes, inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn = nn.ModuleList([nn.BatchNorm2d(inplanes) for i in range(steps)])
        self.relu = nn.ReLU(inplace=True)
        self.steps = steps

        self.shortcut = nn.Conv2d(inplanes, inplanes, kernel_size=3, stride=1, padding=1, bias=False)

        # init the parameter
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def forward(self, x):
        rx = x
        for i in range(self.steps):
            if i == 0:
                z = self.conv(x)
            else:
                z = self.conv(x) + self.shortcut(rx)
            x = self.relu(z)
            x = self.bn[i](x)
        return x


class SAM(nn.Module):
    def __init__(self, bias=False):
        super(SAM, self).__init__()
        self.bias = bias
        self.conv = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=7, stride=1, padding=3, dilation=1,
                              bias=self.bias, padding_mode='circular')

    def forward(self, x):
        max = torch.max(x, 1)[0].unsqueeze(1)
        avg = torch.mean(x, 1).unsqueeze(1)
        concat = torch.cat((max, avg), dim=1)
        output = self.conv(concat)
        output = F.sigmoid(output) * x
        return output


class CAM(nn.Module):
    def __init__(self, channels, r):
        super(CAM, self).__init__()
        self.channels = channels
        self.r = r
        self.linear = nn.Sequential(
            nn.Linear(in_features=self.channels, out_features=self.channels // self.r, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=self.channels // self.r, out_features=self.channels, bias=True))

    def forward(self, x):
        max = F.adaptive_max_pool2d(x, output_size=1)
        avg = F.adaptive_avg_pool2d(x, output_size=1)
        b, c, _, _ = x.size()
        linear_max = self.linear(max.view(b, c)).view(b, c, 1, 1)
        linear_avg = self.linear(avg.view(b, c)).view(b, c, 1, 1)
        output = linear_max + linear_avg
        output = F.sigmoid(output) * x
        return output


class CBAM(nn.Module):
    def __init__(self, channels, r):
        super(CBAM, self).__init__()
        self.channels = channels
        self.r = r
        self.sam = SAM(bias=False)
        self.cam = CAM(channels=self.channels, r=self.r)

    def forward(self, x):
        output = self.cam(x)
        output = self.sam(output)
        return output + x

class SRM(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.cfc = nn.Conv1d(channel, channel, kernel_size=2, groups=channel,
                             bias=False)
        self.bn = nn.BatchNorm1d(channel)

    def forward(self, x):
        b, c, h, w = x.shape
        # style pooling
        mean = x.reshape(b, c, -1).mean(-1).unsqueeze(-1)
        std = x.reshape(b, c, -1).std(-1).unsqueeze(-1)
        u = torch.cat([mean, std], dim=-1)
        # style integration
        z = self.cfc(u)
        z = self.bn(z)
        g = torch.sigmoid(z)
        g = g.reshape(b, c, 1, 1)
        return x * g.expand_as(x)

class GumbelSigmoid(nn.Module):
    """
    Binary-case of Gumbel-Softmax
    https://arxiv.org/pdf/1611.00712.pdf
    """

    def __init__(
        self,
        tau: float = 1.0,
        tau_max: float = 1.0,  # valid if tau is None
        hard: bool = True,
        eps: float = 1e-10,
        pixelwise: bool = True,
    ):
        super().__init__()
        self.tau = tau
        self.tau_max = tau_max
        self.hard = hard
        self.eps = eps
        if self.tau is None:
            self.weight = nn.Parameter(torch.tensor(0.0))
        self.pixelwise = pixelwise
        self.fixed_noise = None

    def logistic_noise(self, logits):
        B, _, H, W = logits.shape
        shape = (B, 1, H, W) if self.pixelwise else (B, 1, 1, 1)
        U1 = torch.rand(*shape, device=logits.device)
        U2 = torch.rand_like(U1)
        l = -torch.log(torch.log(U1 + self.eps) / torch.log(U2 + self.eps) + self.eps)
        return l

    def sigmoid_with_temperature(self, logits):
        if self.tau is None:
            inverse_tau = F.softplus(self.weight) + 1.0 / self.tau_max
            return torch.sigmoid(logits * inverse_tau)
        else:
            return torch.sigmoid(logits / self.tau)

    def forward(self, logits, threshold: float = 0.5):
        if self.fixed_noise is None:
            logits = logits + self.logistic_noise(logits)
        else:
            B, C, H, W = logits.shape
            logits = logits + self.fixed_noise.expand(B, -1, -1, -1)

        mask_soft = self.sigmoid_with_temperature(logits)

        if self.hard:
            # 'mask_hard' outputs and 'mask_soft' gradients
            mask_hard = (mask_soft > threshold).float()
            return mask_hard - mask_soft.detach() + mask_soft
        else:
            return mask_soft

    def extra_repr(self):
        return f"hard={self.hard}, eps={self.eps}"

In [12]:
import torch
import torch.nn as nn
from torch.nn.utils.parametrizations import spectral_norm
import torch.nn.functional as F
import numpy as np


class ResidualBlock(nn.Module):
    """Residual Block with instance normalization."""
    def __init__(self, dim_in, dim_out):
        super(ResidualBlock, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1, bias=False),
            #nn.BatchNorm2d(dim_out,affine=True,track_running_stats=True),
            nn.InstanceNorm2d(dim_out, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(dim_out, dim_out, kernel_size=3, stride=1, padding=1, bias=False),
            #nn.BatchNorm2d(dim_out,affine=True,track_running_stats=True),
            nn.InstanceNorm2d(dim_out, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True)
        )

    def forward(self, x):
        return x + self.main(x)


class Generator_DropDropNet(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=5, repeat_num=5, batch_dim = 8, img_dim = (32,1024)):
        super(Generator_DropDropNet, self).__init__()
        self.gaussianNoise = GaussianNoise(sigma=0.05)

        self.SharedDownSample = nn.Sequential(
            CBAM(1 + c_dim, 2),
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim, 2),
            RCL(conv_dim, 2),

            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim * 2, 2),
            RCL(conv_dim * 2, 2),

            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim * 4, 2),
            RCL(conv_dim * 4, 2),
        )

        self.ResidualDropout = []
        self.ResidualDroplets = []
        for i in range(repeat_num):
            self.ResidualDropout.append(ConvBlock(conv_dim * 4, conv_dim * 4))
            self.ResidualDroplets.append(ConvBlock(conv_dim * 4, conv_dim * 4))

        self.ResidualDropout = nn.Sequential(*self.ResidualDropout)
        self.ResidualDroplets = nn.Sequential(*self.ResidualDroplets)

        self.DropoutUpSample = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Dropout2d(),
            #CBAM(conv_dim * 2, 2),
            #RCL(conv_dim * 2, 2),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(conv_dim * 2, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Dropout2d(),
            #CBAM(conv_dim, 2),
            #RCL(conv_dim, 2),

            nn.Conv2d(conv_dim, 1, kernel_size=7, stride=1, padding=3, bias=False),
        )
        self.gumbel = GumbelSigmoid(hard=True, tau=1, pixelwise=True)

    def forward(self, x, c, get_mask = False):
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))  # c = 8,6,32,2014
        c = self.gaussianNoise(c)
        x_ = torch.cat([x, c], dim=1)           # 8,6,32,1024
        
        x_ = self.SharedDownSample(x_)
        dropout = self.ResidualDropout(x_)
        dropout = self.DropoutUpSample(dropout)
        
        #soft = F.sigmoid(dropout + self.dropout_threshold)
        #hard = (soft > 0.5).float()
        #mask = hard - soft.detach() + soft
        mask = self.gumbel(dropout, 0.5)

        if not get_mask:
            return mask + (1 - mask) * x
        return (mask + (1 - mask) * x, mask.detach())

class ConvBlock(nn.Module):
    def __init__(self, in_planes, out_planes):
        super(ConvBlock, self).__init__()
        self.bn1 = nn.InstanceNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes,int(out_planes / 2), kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.InstanceNorm2d(int(out_planes / 2))
        self.conv2 = nn.Conv2d(int(out_planes / 2), int(out_planes / 4), kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.InstanceNorm2d(int(out_planes / 4))
        self.conv3 = nn.Conv2d(int(out_planes / 4), int(out_planes / 4), kernel_size=3, stride=1, padding=1, bias=False)

        self.downsample = None
        if in_planes != out_planes:
                self.downsample = nn.Sequential(nn.InstanceNorm2d(in_planes),
                                            nn.ReLU(True),
                                            nn.Conv2d(in_planes, out_planes, 1, 1, bias=False))

    def forward(self, x):
        residual = x

        out1 = self.bn1(x)
        out1 = F.relu(out1, True)
        out1 = self.conv1(out1)

        out2 = self.bn2(out1)
        out2 = F.relu(out2, True)
        out2 = self.conv2(out2)

        out3 = self.bn3(out2)
        out3 = F.relu(out3, True)
        out3 = self.conv3(out3)

        out3 = torch.cat((out1, out2, out3), 1)
        if self.downsample is not None:
            residual = self.downsample(residual)
        out3 += residual
        return out3


class Discriminator(nn.Module):
    """Discriminator network with PatchGAN."""

    def __init__(self, image_size=(32, 1024), conv_dim=64, c_dim=5, repeat_num=5):
        super(Discriminator, self).__init__()
        layers = []
        layers.append(nn.Conv2d(1, conv_dim, kernel_size=4, stride=2, padding=1))
        layers.append(nn.LeakyReLU(0.01))

        # layers.append(SRM(conv_dim))

        curr_dim = conv_dim
        for i in range(1, repeat_num):
            layers.append(nn.Conv2d(curr_dim, curr_dim * 2, kernel_size=4, stride=2, padding=1))
            layers.append(nn.LeakyReLU(0.01))
            curr_dim = curr_dim * 2

        kernel_size_x = int(image_size[0] / np.power(2, repeat_num))
        kernel_size_y = int(image_size[1] / np.power(2, repeat_num))
        self.main = nn.Sequential(*layers)
        self.conv1 = nn.Conv2d(curr_dim, 1, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv2 = nn.Conv2d(curr_dim, c_dim, kernel_size=(kernel_size_x, kernel_size_y), bias=False)

    def forward(self, x):
        h = self.main(x)
        out_src = self.conv1(h)
        out_cls = self.conv2(h)
        return out_src, out_cls.view(out_cls.size(0), out_cls.size(1))

In [13]:
##########
## TEST ##
##########

class Generator_UNET(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=5, repeat_num=5, batch_dim = 8, img_dim = (32,1024)):
        super(Generator_UNET, self).__init__()
        self.gaussianNoise = GaussianNoise(sigma=0.05)

        # [B,6,64,1024]
        self.down_1 = nn.Sequential(
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
        )
        # [B,64,32,1024]
        self.down_2 = nn.Sequential(
            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True)
        )
        # [B,128,16,512]
        self.down_3 = nn.Sequential(
            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True)
        )
        # [B,256,8,256]

        self.ResidualDropout = []
        for i in range(repeat_num):
            self.ResidualDropout.append(ConvBlock(conv_dim * 4, conv_dim * 4))

        self.ResidualDropout = nn.Sequential(*self.ResidualDropout)
        
        # [B,256,8,256] + down_3 = [B,512,8,256]
        self.up_1 = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(conv_dim * 8, conv_dim * 4, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Dropout2d(),
        )
        # [B,128,16,512] + down_2 = [B,256,16,512]
        self.up_2 = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim*2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(conv_dim * 2, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Dropout2d(),
            # [B,64,32,1024]
            nn.Conv2d(conv_dim, 1, kernel_size=7, stride=1, padding=3, bias=False),
            # [B,1,32,1024]
        )
        self.gumbel = GumbelSigmoid(hard=True, tau=1, pixelwise=True)

    def forward(self, x, c, get_mask = False):
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))  # c = 8,6,32,2014
        c = self.gaussianNoise(c)
        x_ = torch.cat([x, c], dim=1)           # 8,6,32,1024
        
        x_ = self.down_1(x_)
        x_2 = self.down_2(x_)
        x_3 = self.down_3(x_2)
        residual = self.ResidualDropout(x_3)
        up = torch.cat([residual,x_3],dim=1)
        up = self.up_1(up)
        up = torch.cat([up,x_2], dim=1)
        up = self.up_2(up)

        mask = self.gumbel(up, 0.5)

        if not get_mask:
            return mask + (1 - mask) * x
        return (mask + (1 - mask) * x, mask.detach())

In [14]:
############
## TEST 2 ##
############

class Generator_UNET_NoVerticalReduction(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=5, repeat_num=5, batch_dim = 8, img_dim = (32,1024)):
        super(Generator_UNET_NoVerticalReduction, self).__init__()
        self.gaussianNoise = GaussianNoise(sigma=0.05)

        # [B,6,32,1024]
        self.down_1 = nn.Sequential(
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
        )
        # [B,64,32,1024]
        self.down_2 = nn.Sequential(
            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=(3,4), stride=(1,2), padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True)
        )
        # [B,128,32,512]
        self.down_3 = nn.Sequential(
            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=(3,4), stride=(1,2), padding=1, bias=False),
            nn.InstanceNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True)
        )
        # [B,256,32,256]

        self.ResidualDropout = []
        for i in range(repeat_num):
            self.ResidualDropout.append(ConvBlock(conv_dim * 4, conv_dim * 4))

        self.ResidualDropout = nn.Sequential(*self.ResidualDropout)
        
        # [B,256,32,256] + down_3 = [B,512,32,256]
        self.up_1 = nn.Sequential(
            nn.Upsample(scale_factor=(1,2)),
            nn.Conv2d(conv_dim * 8, conv_dim * 4, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Dropout2d(),
        )
        # [B,128,16,512] + down_2 = [B,256,16,512]
        self.up_2 = nn.Sequential(
            nn.Upsample(scale_factor=(1,2)),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(conv_dim*2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(conv_dim * 2, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            # [B,64,32,1024]
            nn.Conv2d(conv_dim, 1, kernel_size=7, stride=1, padding=3, bias=False),
            # [B,1,32,1024]
        )
        self.gumbel = GumbelSigmoid(hard=True, tau=1, pixelwise=True)

    def forward(self, x, c, get_mask = False, smooth_label = True):
        intensity = torch.max(c).float()
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))  # c = 8,6,32,2014
        if smooth_label:
            c = self.gaussianNoise(c)
        x_ = torch.cat([x, c], dim=1)           # 8,6,32,1024
        
        x_ = self.down_1(x_)
        x_2 = self.down_2(x_)
        x_3 = self.down_3(x_2)
        residual = self.ResidualDropout(x_3)
        up = torch.cat([residual,x_3],dim=1)
        up = self.up_1(up)
        up = torch.cat([up,x_2], dim=1)
        up = self.up_2(up)

        mask = self.gumbel(up, 1.0 - intensity * 0.5)

        if not get_mask:
            return mask + (1 - mask) * x
        return (mask + (1 - mask) * x, mask.detach())

In [15]:
############
## TEST 3 ##
############

class Generator_NoVerticalReduction(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=5, repeat_num=5, batch_dim = 8, img_dim = (32,1024)):
        super(Generator_NoVerticalReduction, self).__init__()
        self.gaussianNoise = GaussianNoise(sigma=0.05)

        self.SharedDownSample = nn.Sequential(
            CBAM(1 + c_dim, 2),
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False, padding_mode='circular'),
            nn.BatchNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim, 2),
            #RCL(conv_dim, 2),

            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=(3,4), stride=(1,2), padding=1, bias=False, padding_mode='circular'),
            nn.BatchNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim * 2, 2),
            #RCL(conv_dim * 2, 2),

            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=(3,4), stride=(1,2), padding=1, bias=False, padding_mode='circular'),
            nn.BatchNorm2d(conv_dim * 4, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim * 4, 2),
            #RCL(conv_dim * 4, 2),
        )

        self.ResidualDropout = []
        self.ResidualDroplets = []
        for i in range(repeat_num):
            self.ResidualDropout.append(ConvBlock(conv_dim * 4, conv_dim * 4))
            self.ResidualDroplets.append(ConvBlock(conv_dim * 4, conv_dim * 4))

        self.ResidualDropout = nn.Sequential(*self.ResidualDropout)
        self.ResidualDroplets = nn.Sequential(*self.ResidualDroplets)

        self.DropoutUpSample = nn.Sequential(
            nn.Upsample(scale_factor=(1,2)),
            nn.Conv2d(conv_dim * 4, conv_dim * 2, kernel_size=7, stride=1, padding=3, bias=False,  padding_mode='circular'),
            nn.BatchNorm2d(conv_dim * 2, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim * 2, 2),
            #RCL(conv_dim * 2, 2),

            nn.Upsample(scale_factor=(1,2)),
            nn.Conv2d(conv_dim * 2, conv_dim, kernel_size=7, stride=1, padding=3, bias=False,  padding_mode='circular'),
            nn.BatchNorm2d(conv_dim, eps=1e-5,momentum=0.1,affine=True,track_running_stats=True),
            nn.LeakyReLU(inplace=True),
            CBAM(conv_dim, 2),
            #RCL(conv_dim, 2),

            nn.Conv2d(conv_dim, 1, kernel_size=7, stride=1, padding=3, bias=False,  padding_mode='circular'),
        )
        self.gumbel = GumbelSigmoid(hard=True, tau=1, pixelwise=True)

    def forward(self, x, c, get_mask = False, smooth_label = True):
        intensity = torch.max(c).float()
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))  # c = 8,6,32,2014
        if smooth_label:
            c = self.gaussianNoise(c)
        x_ = torch.cat([x, c], dim=1)           # 8,6,32,1024
        
        x_ = self.SharedDownSample(x_)
        dropout = self.ResidualDropout(x_)
        dropout = self.DropoutUpSample(dropout)
        
        #soft = F.sigmoid(dropout - x)
        #hard = (soft > 1.0 - intensity * 0.5).float()
        #mask = hard - soft.detach() + soft
        
        #mask = self.gumbel(dropout, 0.5)
        mask = self.gumbel(dropout + x, 1.0 - intensity * 0.5)
        #mask = self.gumbel(dropout - x, intensity * 0.5)
        
        if not get_mask:
            #return mask * x
            return mask + (1 - mask) * x
        #return  mask * x, mask.detach()
        return (mask + (1 - mask) * x, mask.detach())

In [16]:
from skimage import io
import random

import torch
import torch.nn.functional as F
from torch import nn
from torchvision import transforms
import numpy as np
import os
import time
import datetime
from tqdm import tqdm

class RayGAN(object):

    def __init__(self, data_loader, config):
        PP_good("RAY GAN!")

        # Parameters Initialization
        self.data_loader = data_loader
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.image_size = config.image_size
        self.c_dim = config.c_dim
        self.g_conv_dim = config.g_conv_dim
        self.d_conv_dim = config.d_conv_dim
        self.g_repeat_num = config.g_repeat_num
        self.d_repeat_num = config.d_repeat_num
        self.lambda_cls = config.lambda_cls
        self.lambda_rec = config.lambda_rec
        self.lambda_identity = config.lambda_identity

        # Training Initialization
        self.batch_size = config.batch_size
        self.num_epoch = config.num_epoch
        self.num_iters_decay = config.num_iters_decay
        self.g_lr = config.g_lr
        self.d_lr = config.d_lr
        self.n_critic = config.n_critic
        self.beta1 = config.beta1
        self.beta2 = config.beta2
        self.lr_update_step = config.lr_update_step
        self.lr_decay_multiplier = config.lr_decay_multiplier
        self.lr_decay_after_epochs = config.lr_decay_after_epochs
        self.lr_decay_stop_after_epochs = config.lr_decay_stop_after_epochs

        # Save and Load
        self.load = config.load_checkpoint
        self.save = config.save_checkpoint
        self.checkpoint_prefix = config.checkpoint_prefix
        self.save_iters = config.save_iters
        self.save_dir = config.save_dir
        self.loaded_epoch = 0

        # Net Initialization
        self.g = Generator_NoVerticalReduction(self.g_conv_dim, self.c_dim, self.g_repeat_num, self.batch_size, self.image_size)
        self.d = Discriminator(self.image_size, self.d_conv_dim, self.c_dim, self.d_repeat_num)
        self.g_optimizer = torch.optim.Adam(self.g.parameters(), self.g_lr, (self.beta1, self.beta2), weight_decay=1e-5)
        self.d_optimizer = torch.optim.Adam(self.d.parameters(), self.d_lr, (self.beta1, self.beta2), weight_decay=1e-5)
        self.g.to(self.device)
        self.d.to(self.device)
        self.g = self.g.apply(self.init_weights)
        self.d = self.d.apply(self.init_weights)
        #self.ssim_loss = SSIM(win_size=11, win_sigma=1.5, data_range=1, size_average=True, channel=1)

        if self.load:
            self.load_latest_checkpoint()

    def init_weights(self, layer):
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.ConvTranspose2d):
            nn.init.normal_(layer.weight, 0.0, 0.02)
        if isinstance(layer, nn.BatchNorm2d): # or isinstance(layer, nn.InstanceNorm2d):
            nn.init.normal_(layer.weight, 0.0, 0.02)
            nn.init.constant_(layer.bias, 0)

    def train(self):
        """ Train the RayGAN """
        # CACHE LEARNING RATE FOR DECAYING
        g_lr = self.g_lr
        d_lr = self.d_lr

        # TRAINING LOOP
        print("Start training on {}".format(self.device))
        start_time = time.time()

        data_iter = iter(self.data_loader)

        g_losses = []
        d_losses = []

        total_iteration = 0
        for epoch in range(self.num_epoch):
            for i, next_sample in tqdm(enumerate(self.data_loader), total=len(self.data_loader)):
                #for i in range(0, self.num_iters):

                # Fetch next sample ['weather_pgm','synth_pgm','weather_label','synth_label']
                # synth labels are always [0.0, 0.0, ... , 0.0, 1.0] tensors
                # weather_pgm and synth_pgm are:
                # [B, 1, W, H] where 1 is the only gray-scale channel
                # try:
                #     next_sample = next(data_iter)
                # except:
                #     data_iter = iter(self.data_loader)
                #     next_sample = next(data_iter)

                weather_pgm = next_sample['weather_pgm'].to(self.device)
                weather_labels = next_sample['weather_label'][:,:-1].to(self.device)
                synth_pgm = next_sample['synth_pgm'].to(self.device)
                synth_label = next_sample['synth_label'].to(self.device)
                target_weather_labels = weather_labels #self.generate_target_labels(synth_label).to(self.device)

                # ============================================= #
                # =============== DISCRIMINATOR =============== #
                # ============================================= #

                # Loss on real weather
                d_real_realism, d_real_cls = self.d(weather_pgm)
                # la funzione F.cross_entropy usa i logits (non le probabilità), quindi va bene in quanto il Discr non ha sigmoid alla fine.
                d_cls_loss = F.cross_entropy(d_real_cls, torch.argmax(weather_labels, dim=1))
                # d_loss_real = - torch.mean(d_real_realism)

                # Loss on generated weather
                generated_pgm = self.g(synth_pgm.float(), target_weather_labels.float())
                d_fake_realism, d_fake_cls = self.d(generated_pgm.detach())
                # d_loss_fake = torch.mean(d_fake_realism)

                # Relativistic loss
                #  MA MANCA UN PEZZO?!??! GUARDA eq. (25) nell'appendice di in https://arxiv.org/pdf/1807.00734.pdf
                # Forse manca:
                # + F.binary_cross_entropy_with_logits(d_fake_realism - torch.mean(d_real_realism), torch.zeros_like(d_real_realism))
                d_ra_loss = F.binary_cross_entropy_with_logits(
                     d_real_realism - torch.mean(d_fake_realism),
                     torch.ones_like(d_real_realism)) + F.binary_cross_entropy_with_logits(
                    d_fake_realism - torch.mean(d_real_realism), torch.zeros_like(d_real_realism))

                # d_ra_loss = (torch.mean(torch.nn.ReLU()(1.0 - (d_real_realism - torch.mean(d_fake_realism)))) +
                #              + torch.mean(torch.nn.ReLU()(1.0 + (d_fake_realism - torch.mean(d_real_realism))))) / 2.0
                
                #d_ra_loss = F.binary_cross_entropy_with_logits(d_real_realism, torch.ones_like(d_real_realism)) + F.binary_cross_entropy_with_logits(d_fake_realism, torch.zeros_like(d_fake_realism))

                # Backward
                d_loss = d_ra_loss + self.lambda_cls * d_cls_loss
                d_losses.append(d_loss.to('cpu').detach().numpy())
                self.d_optimizer.zero_grad()
                d_loss.backward()
                self.d_optimizer.step()

                # ============================================= #
                # ================= GENERATOR ================= #
                # ============================================= #
                if i % self.n_critic == 0:

                    generated_pgm,mask = self.g(synth_pgm.float(), target_weather_labels.float(), get_mask = True)
                    g_fake_realism, g_fake_cls = self.d(generated_pgm)
                    g_loss_cls = F.cross_entropy(g_fake_cls,torch.argmax(target_weather_labels.float(), dim=1))
                    #g_real_realism, _ = #self.d(weather_pgm.detach())
                    g_real_realism = d_real_realism.detach()

                    # g_loss_cycle = torch.mean(torch.abs(synth_pgm.float() - self.g(generated_pgm, synth_label.float())))
                    # g_loss_cycle = torch.mean(torch.abs(synth_pgm.float() - self.g(generated_pgm, torch.zeros_like(weather_labels,requires_grad=False,dtype=torch.float))))
                    # g_loss_cycle = 1 - self.ssim_loss(synth_pgm.float(), self.g(generated_pgm, synth_label.float()))
                    # g_loss_cycle = 1 - self.ssim_loss(synth_pgm.float(), self.g(generated_pgm, zeros_like(weather_label,requires_grad=false,dtype=torch.float)))

                    g_ra_loss = F.binary_cross_entropy_with_logits(
                        g_fake_realism - torch.mean(g_real_realism),
                        torch.ones_like(g_real_realism)) + F.binary_cross_entropy_with_logits(
                        g_real_realism - torch.mean(g_fake_realism),
                        torch.zeros_like(g_real_realism)
                    )
                    
                    #g_ra_loss = F.binary_cross_entropy_with_logits(g_fake_realism, torch.ones_like(g_fake_realism))
                    #g_ra_loss = (torch.mean(torch.nn.ReLU()(1.0 + (g_real_realism.detach() - torch.mean(g_fake_realism)))) +
                    #             + torch.mean(torch.nn.ReLU()(1.0 - (g_fake_realism - torch.mean(g_real_realism))))) / 2.0

                    # g_ra_loss = 0.5 * (torch.mean(g_real_realism.detach() - torch.mean(g_fake_realism) + 1.0)**2) + 0.5 * (torch.mean(g_fake_realism - torch.mean(g_real_realism.detach()) - 1.0))

                    #g_real_identity = self.g(weather_pgm.float(), weather_labels.float())
                    #g_real_identity_loss = torch.mean(torch.abs(weather_pgm.float() - g_real_identity))

                    # Backward
                    #g_loss = 1.0 * g_ra_loss + self.lambda_cls * g_loss_cls + self.lambda_identity * g_real_identity_loss
                    g_loss = 1.0 * g_ra_loss + self.lambda_cls * g_loss_cls
                    #g_loss = 1.0 * g_ra_loss + self.lambda_cls * g_loss_cls + self.lambda_rec * g_loss_cycle
                    #g_loss = 1.0 * g_ra_loss + self.lambda_cls * g_loss_cls + self.lambda_identity * g_real_identity_loss # + self.lambda_rec * g_loss_cycle 
                    g_losses.append(g_loss.to('cpu').detach().numpy())
                    self.g_optimizer.zero_grad()
                    g_loss.backward()
                    self.g_optimizer.step()

                    if i % 100 == 0:
                        print('\nD LOSS: {}\nG LOSS: {}'.format(np.mean(d_losses), np.mean(g_losses)))
                        print('G_LOSS_CLS: {}\nG_RA_LOSS:{}\n'.format(g_loss_cls, g_ra_loss))
                        d_losses = []
                        g_losses = []
                        #A = torch.cat((synth_pgm[:4, :, :, :], weather_pgm[:4, :, :, :]), dim=0)
                        #B = torch.cat((target_weather_labels[:4, :], weather_labels[:4, :]), dim=0)
                        #G = torch.cat((generated_pgm[:4, :, :, :], g_real_identity[:4, :, :, :]), dim=0)
                        A = synth_pgm[:9,:,:,:]
                        B = target_weather_labels[:9,:]
                        G = generated_pgm[:9,:,:,:]
                        M = mask[:9, :, :, :]
                        save_synth_and_generated(A, G, B, M, os.path.join('/kaggle/working/Results', 'test_{}_{}'.format(epoch, i)), 2)
                        # save_synth_and_generated(synth_pgm, generated_pgm, target_weather_labels, os.path.join('Results', 'test_{}_{}'.format(epoch, i)), 2)

                # Decay learning rates.
                if (epoch + self.loaded_epoch) > self.lr_decay_after_epochs and (epoch + self.loaded_epoch) < self.lr_decay_stop_after_epochs:                    
                    if (total_iteration+1) % self.lr_update_step == 0 and (total_iteration + 1) > self.num_iters_decay:
                        #g_lr -= (self.g_lr / float(self.num_iters_decay))
                        #d_lr -= (self.d_lr / float(self.num_iters_decay))
                        g_lr *= self.lr_decay_multiplier
                        d_lr *= self.lr_decay_multiplier
                        self.update_lr(g_lr, d_lr)
                        print('Decayed learning rates, g_lr: {}, d_lr: {}.\n'.format(g_lr, d_lr))

                if self.save and total_iteration % self.save_iters == 0 and total_iteration > 0:
                    self.save_checkpoint(epoch + self.loaded_epoch, i, total_iteration, g_lr, d_lr)

                total_iteration += 1

    def testG(self, pgm_name):
        self.load_latest_checkpoint()
        device = 'cpu'
        if torch.cuda.is_available():
            device = 'cuda'
        
        synth_pgm_name = os.path.join(pgm_name)
        synth_pgm = io.imread(synth_pgm_name)
        batch = {}
        batch['synth_pgm'] = torch.zeros((self.batch_size, 1, self.image_size[0], self.image_size[1])).to(device)
        batch['weather_pgm'] = torch.zeros((self.batch_size, 1, self.image_size[0], self.image_size[1])).to(device)
        batch['weather_label'] = torch.zeros((self.batch_size, self.c_dim)).to(device)
        batch['synth_label'] = torch.zeros((self.batch_size, self.c_dim)).to(device)
        
        t = transforms.ToTensor()
        batch['synth_pgm'][0] = t(synth_pgm)
        batch['weather_label'][0][0] = 0.8

        #synth_label = torch.zeros((self.batch_size, self.c_dim))
        #synth_label[0][-1] = 1
        #c = self.generate_target_labels(synth_label)
        #c[0] *= 0.0
        #c[0][4] = 1.0
        #print(c)
        #c = torch.zeros((self.batch_size, self.c_dim)).to(device)
        #target_label = self.generate_target_labels(batch['weather_label'])
        generated, mask = self.g(batch['synth_pgm'], batch['weather_label'], get_mask=True)
        sample = {}
        sample['weather_pgm'] = generated[0]
        sample['synth_pgm'] = t(synth_pgm)
        sample['weather_label'] = batch['weather_label'][0]
        show_sample(sample)
        #save_synth_and_generated(batch['synth_pgm'],generated,target_label,mask,'TEST',2)
        #_,classification = self.d(generated)
        #print(target_label)
        #print(target_label.dtype)
        #print(classification.shape)
        #print(F.cross_entropy(classification,target_label))
        #print(F.cross_entropy(classification,torch.argmax(target_label, dim=1)))
        
    def testD(self, pgm_name):
        self.load_latest_checkpoint()
        device = 'cpu'
        if torch.cuda.is_available():
            device = 'cuda'
        
        pgm_name = os.path.join(pgm_name)
        pgm = io.imread(pgm_name)
        batch = {}
        batch['synth_pgm'] = torch.zeros((self.batch_size, 1, self.image_size[0], self.image_size[1])).to(device)
        batch['weather_label'] = torch.zeros((self.batch_size, self.c_dim)).to(device)
        
        t = transforms.ToTensor()
        batch['synth_pgm'][:] = t(pgm)
        _,classification = self.d(batch['synth_pgm']) 
        print(classification)

    def generate_target_labels(self, synth_labels):
        """
        :param synth_labels: [B, num_possible_label] where num_possible_labels are weather_labels + 1 (synth)
        :return: [B, num_possible_label] with random one hot selected weather to generate
        """
        batch_size = synth_labels.size(0)
        target_labels = torch.zeros_like(synth_labels)
        random_int = torch.randint(0, synth_labels.size(1) - 1, size=(batch_size, 1)).squeeze()
        target_labels[np.arange(batch_size), random_int] = 1.0
        return target_labels

    def update_lr(self, g_lr, d_lr):
        """Decay learning rates of the generator and discriminator."""
        for param_group in self.g_optimizer.param_groups:
            param_group['lr'] = g_lr
        for param_group in self.d_optimizer.param_groups:
            param_group['lr'] = d_lr

    def save_checkpoint(self, curr_epoch, curr_iteration, total_iterations,lr_g, lr_d):
        saving_struct = {}
        saving_struct['G'] = self.g.state_dict()
        saving_struct['D'] = self.d.state_dict()
        saving_struct['G_opt'] = self.g_optimizer.state_dict()
        saving_struct['D_opt'] = self.d_optimizer.state_dict()
        saving_struct['G_lr'] = lr_g
        saving_struct['D_lr'] = lr_d
        saving_struct['last_epoch'] = curr_epoch + 1
        path = f'{self.save_dir}{self.checkpoint_prefix}_{curr_epoch}x{curr_iteration}_{total_iterations}.pth'
        torch.save(saving_struct, path)
        PP_good(f'SAVED in {path}')

    def load_latest_checkpoint(self):
        files = os.listdir(self.save_dir)
        files = [file for file in files if file.startswith(self.checkpoint_prefix)]
        files.sort()

        if len(files) == 0:
            PP_warning("No checkpoint found.")
            return

        #path = f'{self.save_dir}{files[-1]}'
        path = "/kaggle/working/Saves/Ray_4x1727_4000.pth"
        device = 'cpu'
        if torch.cuda.is_available():
            device = 'cuda'
        loading_struct = torch.load(path, map_location=torch.device(device))

        self.g.load_state_dict(loading_struct['G'])
        self.d.load_state_dict(loading_struct['D'])
        self.g_optimizer.load_state_dict(loading_struct['G_opt'])
        self.d_optimizer.load_state_dict(loading_struct['D_opt'])
        self.g_lr = loading_struct['G_lr']
        self.d_lr = loading_struct['D_lr']
        self.loaded_epoch = loading_struct['last_epoch']
        PP_good(f'LOADED {path}.\nNew lr_G = {self.g_lr}\nNew lr_D = {self.d_lr}')


In [ ]:
### import torch
import argparse

# sample:   [B,1,32,1024]
#           B: num of samples in a batch
#           1: just one channel (gray-scale image)
#           1024: width
#           32: height
#
# 1: SUN - 2: RAIN - 3: FOG - 4: SNOW - 5: SYN
# label:    [B,5]
#           B: num of samples in a batch
#           5: one-hot label encoding => [0,0,1,0,0] sample with fog

parser = argparse.ArgumentParser()

# Model configuration.
parser.add_argument('--c_dim', type=int, default=4, help='dimension of domain labels')
parser.add_argument('--g_conv_dim', type=int, default=64, help='number of conv filters in the first layer of G')
parser.add_argument('--d_conv_dim', type=int, default=64, help='number of conv filters in the first layer of D')
parser.add_argument('--g_repeat_num', type=int, default=5, help='number of residual blocks in G')
parser.add_argument('--d_repeat_num', type=int, default=3, help='number of strided conv layers in D')
parser.add_argument('--lambda_cls', type=float, default=1, help='weight for domain classification loss')
parser.add_argument('--lambda_rec', type=float, default=1, help='weight for reconstruction loss')
parser.add_argument('--lambda_identity', type=float, default=0.5, help='weight for identity loss')
parser.add_argument('--image_size', type=tuple, default=(32, 1024), help='image resolution')

# Training configuration.
parser.add_argument('--batch_size', type=int, default=16, help='mini-batch size') # 32 !!!
parser.add_argument('--num_epoch', type=int, default=10, help='number of epochs')
parser.add_argument('--num_iters_decay', type=int, default=4000, help='number of iterations for decaying lr')
parser.add_argument('--g_lr', type=float, default=0.00001, help='learning rate for G')
parser.add_argument('--d_lr', type=float, default=0.00001, help='learning rate for D')
parser.add_argument('--n_critic', type=int, default=1, help='number of D updates per each G update')
parser.add_argument('--beta1', type=float, default=0.5, help='beta1 for Adam optimizer')
parser.add_argument('--beta2', type=float, default=0.999, help='beta2 for Adam optimizer')
# Save and Load
parser.add_argument('--load_checkpoint', type=bool, default=True, help='load a pre trained model')
parser.add_argument('--save_checkpoint', type=bool, default=True, help='save the current model')
parser.add_argument('--checkpoint_prefix', type=str, default='Ray', help='the prefix of the checkpoint')
parser.add_argument('--save_iters', type=int, default=400, help='save the model after x iterations')
parser.add_argument('--save_dir', type=str, default='/kaggle/working/Saves/', help='saving directory')

# Test configuration.
parser.add_argument('--test_iters', type=int, default=200000, help='test model from this step')

# Miscellaneous.
parser.add_argument('--num_workers', type=int, default=1)
parser.add_argument('--mode', type=str, default='train', choices=['train', 'test'])

# Directories.
parser.add_argument('--csv_file', type=str, default='/kaggle/input/weatherpgms/Dataset/dataset.csv')
parser.add_argument('--pgm_dir', type=str, default='/kaggle/input/weatherpgms/Dataset/PGM/')

# Step size.
parser.add_argument('--log_step', type=int, default=10)
parser.add_argument('--sample_step', type=int, default=1000)
parser.add_argument('--model_save_step', type=int, default=10000)
parser.add_argument('--lr_update_step', type=int, default=200)
parser.add_argument('--lr_decay_after_epochs', type=int, default=3)
parser.add_argument('--lr_decay_stop_after_epochs', type=int, default=7)
parser.add_argument('--lr_decay_multiplier', type=int, default=0.6) # 1.0 = no decay

###########
# RAY GAN #
###########
if __name__ == '__main__':
    config = parser.parse_known_args()[0]
    data_loader = []
    if config.mode == 'train':
        data_loader = get_dataloader(config)
    #print(torch.cuda.is_available())
    r = RayGAN(data_loader, config)
    r.train()
    #r.testG('/kaggle/input/weatherpgms/Dataset/PGM/synth_1000.png')
    #r.testD('/kaggle/input/weatherpgms/Dataset/PGM_Test.png')

# ____ For using GENERATOR ____
# x = torch.randn(size=[B, 1, 32, 1024])
# c = torch.randn(size=[B, 5])
# c.requires_grad = False
# x.requires_grad = False
# G = Generator()
# r = G(x, c)   # [B,1, 32, 1024]
# ____ For using DISCRIMINATOR ____
# x = torch.randn(size=[B, 1, 32, 1024])
# x.requires_grad = False
# D = Discriminator(image_size=(x.shape[2], x.shape[3]))
# r = D(x)
# r[0].shape real/fake patches [B,1, number_horizontal_patches ,number_vertical_patches]
# r[1].shape weather labels prediction [B,5]


Checking dataset...
CSV and PGM are correct.
The dataset is unbalanced!
For better training, balance the dataset.
Found 36370 real weather and 12960 synth pgm!
RAY GAN!
LOADED /kaggle/working/Saves/Ray_4x1727_4000.pth.
New lr_G = 1e-05
New lr_D = 1e-05
Start training on cuda


  0%|          | 0/2273 [00:00<?, ?it/s]


D LOSS: 0.2960634231567383
G LOSS: 6.111970901489258
G_LOSS_CLS: 0.11810853332281113
G_RA_LOSS:5.993862152099609



  2%|▏         | 40/2273 [00:56<49:51,  1.34s/it] 